In [ ]:
# ==============================================================================
# BƯỚC 1: CÀI ĐẶT THƯ VIỆN
# ==============================================================================
!pip install pandas_ta hmmlearn -q

import requests
import time
import pandas as pd
import pandas_ta as ta
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from hmmlearn.hmm import GaussianHMM
import warnings
warnings.filterwarnings("ignore")

# ==============================================================================
# BƯỚC 2: CẤU HÌNH CHIẾN LƯỢC (HMM + ICHIMOKU HYBRID)
# ==============================================================================
CONFIG = {
    'SYMBOL': 'BTCUSDT',
    'INTERVAL': '1h',        # Khung thời gian cơ sở (Execution)
    'HTF_INTERVAL': '4h',    # Khung thời gian HMM (Regime Filter)

    'START_DATE': '01/01/2021',
    'END_DATE': '31/12/2025',
    'INITIAL_CAPITAL': 10000.0,

    # --- HMM Settings ---
    'HMM_WINDOW': 200,       # Cửa sổ huấn luyện Rolling
    'N_COMPONENTS': 3,       # 3 trạng thái: Bull, Bear, Sideway
    'HMM_ATR_PERIOD': 14,    # Feature 2 cho HMM

    # --- Ichimoku Settings (Standard) ---
    'TENKAN': 9,
    'KIJUN': 26,
    'SENKOU_B': 52,
    'DISPLACEMENT': 26,

    # --- Risk Management ---
    'SL_ATR_MULT': 1.0,      # SL cách mây 1 ATR
    'RISK_PER_TRADE': 0.02,  # Risk 2% mỗi lệnh (hoặc fixed volume)

    # Costs
    'FEE_RATE': 0.0004,
    'SLIPPAGE': 0.0002
}

# ==============================================================================
# BƯỚC 3: MODULE TẢI DỮ LIỆU
# ==============================================================================
def str_to_millis(date_str: str) -> int:
    dt = datetime.strptime(date_str, "%d/%m/%Y")
    return int(dt.timestamp() * 1000)

def fetch_binance_kline_df(symbol: str, interval: str, start_str: str, end_str: str) -> pd.DataFrame:
    # (Sử dụng lại hàm fetch tối ưu từ các bài trước)
    base_url = "https://data-api.binance.vision/"
    url = base_url + "api/v3/klines"
    req_symbol = symbol.upper()

    # Check connect & switch to .us if needed (Code rút gọn cho đỡ dài)
    try:
        test = requests.get(f"{base_url}/api/v3/time", timeout=5)
        if test.status_code == 451:
             base_url = "https://api.binance.us"
             if req_symbol == "BTCUSDT": req_symbol = "BTCUSD"
    except:
         base_url = "https://api.binance.us"
         req_symbol = "BTCUSD"

    url = base_url + "/api/v3/klines"
    start_time = str_to_millis(start_str)

    if end_str == datetime.now().strftime("%d/%m/%Y"): end_time = int(time.time() * 1000)
    else: end_dt = datetime.strptime(end_str, "%d/%m/%Y").replace(hour=23, minute=59, second=59); end_time = int(end_dt.timestamp() * 1000)

    all_klines = []
    print(f"⏳ Downloading {req_symbol} ({interval})...")

    while start_time < end_time:
        params = {"symbol": req_symbol, "interval": interval, "startTime": start_time, "endTime": end_time, "limit": 1000}
        try:
            r = requests.get(url, params=params, timeout=10)
            if r.status_code != 200: break
            data = r.json();
            if not data: break
            all_klines.extend(data)
            start_time = data[-1][0] + 1
            time.sleep(0.05)
        except: break

    if not all_klines: return pd.DataFrame()
    df = pd.DataFrame(all_klines, columns=["timestamp", "Open", "High", "Low", "Close", "Volume", "CT", "QAV", "NT", "TBBV", "TBQV", "I"])
    df = df[["timestamp", "Open", "High", "Low", "Close", "Volume"]]
    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")
    df.set_index("timestamp", inplace=True)
    df = df.astype(float)
    return df

# ==============================================================================
# BƯỚC 4: HMM REGIME DETECTOR (CORE AI MODULE)
# ==============================================================================
def train_hmm_rolling(df_htf, window_size=200):
    """
    Huấn luyện HMM theo cơ chế Rolling Window để tránh Look-ahead Bias.
    Trả về chuỗi State (0, 1, 2) cho toàn bộ dataframe.
    """
    print("🧠 Training HMM Model (Rolling Window)... Điều này có thể mất vài phút.")

    # 1. Feature Engineering (HTF)
    # Feature 1: Log Returns
    df_htf['Log_Ret'] = np.log(df_htf['Close'] / df_htf['Close'].shift(1))
    # Feature 2: ATR (Biến động)
    df_htf['ATR'] = ta.atr(df_htf['High'], df_htf['Low'], df_htf['Close'], length=14)
    # Normalize ATR (để cùng scale với Log Returns)
    df_htf['ATR_Norm'] = (df_htf['ATR'] - df_htf['ATR'].rolling(window_size).mean()) / df_htf['ATR'].rolling(window_size).std()

    df_htf.dropna(inplace=True)

    # 2. Rolling Prediction
    states_pred = np.full(len(df_htf), -1) # -1 là chưa có dữ liệu

    # Khởi tạo model HMM
    # n_components=3 (Bear, Bull, Sideway)
    model = GaussianHMM(n_components=3, covariance_type="full", n_iter=100, random_state=42)

    data = df_htf[['Log_Ret', 'ATR_Norm']].values

    # Chỉ bắt đầu predict khi đủ window_size dữ liệu
    # Để tăng tốc Backtest, ta sẽ re-train model mỗi 24 nến (1 ngày HTF) thay vì mỗi nến
    # Nếu muốn chính xác 100% thì retrain mỗi nến (rất chậm)
    retrain_step = 1

    for t in range(window_size, len(df_htf), retrain_step):
        # Lấy dữ liệu cửa sổ quá khứ [t-window : t]
        train_data = data[t-window_size : t]

        try:
            model.fit(train_data)

            # Dự đoán trạng thái của nến hiện tại (t)
            # Lưu ý: model.predict trả về chuỗi state cho cả train_data, ta lấy cái cuối cùng
            current_state = model.predict(train_data)[-1]

            # --- MAP STATE LOGIC ---
            # HMM trả về 0,1,2 ngẫu nhiên (không biết số nào là Tăng/Giảm).
            # Ta cần Map lại dựa trên Mean Return của State đó.
            means = model.means_[:, 0] # Lấy trung bình Log Return của 3 state
            sorted_idx = np.argsort(means) # Sắp xếp tăng dần: [Bear, Sideway, Bull]

            state_map = {
                sorted_idx[0]: 2, # Mean thấp nhất -> Bear (State 2)
                sorted_idx[1]: 0, # Mean ở giữa -> Sideway (State 0)
                sorted_idx[2]: 1  # Mean cao nhất -> Bull (State 1)
            }

            mapped_state = state_map[current_state]
            states_pred[t] = mapped_state

        except:
            states_pred[t] = 0 # Mặc định Sideway nếu lỗi

        if t % 500 == 0: print(f"   -> Processed {t}/{len(df_htf)} bars...")

    df_htf['HMM_State'] = states_pred
    return df_htf[['HMM_State']]

# ==============================================================================
# BƯỚC 5: BACKTEST ENGINE (Hybrid Logic)
# ==============================================================================
def run_hybrid_strategy(df_1h, df_4h_states):
    print("⚔️ Running Hybrid Strategy...")
    c = CONFIG

    # --- A. MERGE DATA (MTF) ---
    # Merge HMM State từ khung 4H sang khung 1H
    # ffill để nến 1h biết trạng thái của nến 4h đang bao trùm nó
    df = df_1h.join(df_4h_states, how='left')
    df['HMM_State'] = df['HMM_State'].fillna(method='ffill').fillna(0) # Fill NaN

    # --- B. INDICATORS (ICHIMOKU 1H) ---
    # 1. Tenkan, Kijun
    high_9 = df['High'].rolling(c['TENKAN']).max(); low_9 = df['Low'].rolling(c['TENKAN']).min()
    df['Tenkan'] = (high_9 + low_9) / 2

    high_26 = df['High'].rolling(c['KIJUN']).max(); low_26 = df['Low'].rolling(c['KIJUN']).min()
    df['Kijun'] = (high_26 + low_26) / 2

    # 2. Senkou Span A, B (Cloud) - Shifted Forward 26
    # Để biết giá hiện tại so với Mây, ta cần giá trị mây ĐANG Ở HIỆN TẠI (được vẽ từ quá khứ)
    # SpanA_Current[i] = (Tenkan[i-26] + Kijun[i-26]) / 2
    df['SpanA'] = ((df['Tenkan'] + df['Kijun']) / 2).shift(c['DISPLACEMENT'])

    high_52 = df['High'].rolling(c['SENKOU_B']).max(); low_52 = df['Low'].rolling(c['SENKOU_B']).min()
    df['SpanB'] = ((high_52 + low_52) / 2).shift(c['DISPLACEMENT'])

    # 3. ATR cho SL
    df['ATR'] = ta.atr(df['High'], df['Low'], df['Close'], length=14)

    df.dropna(inplace=True)

    # --- C. EXECUTION LOOP ---
    capital = c['INITIAL_CAPITAL']
    position = 0; entry_price = 0.0; sl_price = 0.0
    trades = []; equity_curve = [capital] * len(df)

    # Numpy convert
    O = df['Open'].values; H = df['High'].values; L = df['Low'].values; C = df['Close'].values
    Times = df.index
    Kijun = df['Kijun'].values
    SpanA = df['SpanA'].values; SpanB = df['SpanB'].values
    HMM = df['HMM_State'].values
    ATR = df['ATR'].values

    for i in range(1, len(df)):
        current_state = HMM[i] # 0: Sideway, 1: Bull, 2: Bear

        # 1. QUẢN LÝ LỆNH ĐANG MỞ (EXIT & TRAILING)
        if position != 0:
            exit_signal = False; exit_reason = ""

            # --- EMERGENCY EXIT (HMM Change) ---
            # Nếu đang Buy mà HMM chuyển sang Bear/Sideway -> Thoát
            if position == 1 and current_state != 1:
                exit_signal = True; exit_reason = f"HMM Change ({current_state})"
            # Nếu đang Sell mà HMM chuyển sang Bull/Sideway -> Thoát
            elif position == -1 and current_state != 2:
                exit_signal = True; exit_reason = f"HMM Change ({current_state})"

            # --- ICHIMOKU EXIT (Kijun Cross) ---
            # Buy: Giá đóng cửa dưới Kijun
            if position == 1 and C[i] < Kijun[i]:
                 exit_signal = True; exit_reason = "Kijun Break"
            # Sell: Giá đóng cửa trên Kijun
            elif position == -1 and C[i] > Kijun[i]:
                 exit_signal = True; exit_reason = "Kijun Break"

            # --- STOP LOSS ---
            if position == 1 and L[i] <= sl_price:
                exit_signal = True; exit_reason = "SL Hit"
            elif position == -1 and H[i] >= sl_price:
                exit_signal = True; exit_reason = "SL Hit"

            # EXECUTE EXIT
            if exit_signal:
                if position == 1:
                    real_exit = min(O[i], sl_price) if "SL" in exit_reason else C[i]
                    pnl = (real_exit * (1 - c['SLIPPAGE']) - entry_price) / entry_price
                else:
                    real_exit = max(O[i], sl_price) if "SL" in exit_reason else C[i]
                    pnl = (entry_price - real_exit * (1 + c['SLIPPAGE'])) / entry_price

                net_pnl = pnl - (c['FEE_RATE'] * 2)
                capital += (capital * net_pnl) # Fixed Balance (hoặc Compounding)

                trades.append({'Date': Times[i], 'Type': position, 'Reason': exit_reason, 'PnL': capital*net_pnl, 'Equity': capital})
                position = 0

            else:
                # --- TRAILING STOP (Bám theo Kijun) ---
                if position == 1:
                    # Kijun chỉ tăng hoặc đi ngang trong Uptrend (không dời xuống)
                    if Kijun[i] > sl_price: sl_price = Kijun[i]
                elif position == -1:
                    if Kijun[i] < sl_price: sl_price = Kijun[i]

        # 2. TÌM ĐIỂM VÀO LỆNH (ENTRY TRIGGER)
        if position == 0:

            # --- SETUP LONG ---
            if current_state == 1: # HMM Bullish
                # Condition 1: Giá nằm trên Mây (Span A và Span B)
                cloud_top = max(SpanA[i], SpanB[i])
                if C[i] > cloud_top:
                    # Condition 2: Pullback chạm Kijun (Low chạm Kijun nhưng Close vẫn giữ được hoặc bật lên)
                    # Logic đơn giản hóa: Giá đóng cửa > Kijun VÀ Giá thấp nhất <= Kijun (Chạm bật)
                    # Hoặc giá vừa cắt lên Kijun
                    pullback_kijun = (L[i] <= Kijun[i]) and (C[i] > Kijun[i])

                    if pullback_kijun:
                        position = 1
                        entry_price = C[i] * (1 + c['SLIPPAGE'])
                        # SL = Đáy mây - 1 ATR (hoặc Kijun - ATR tùy khẩu vị)
                        # Theo thiết kế: Mép dưới mây - 1 ATR
                        cloud_bottom = min(SpanA[i], SpanB[i])
                        sl_price = cloud_bottom - (c['SL_ATR_MULT'] * ATR[i])

            # --- SETUP SHORT ---
            elif current_state == 2: # HMM Bearish
                # Condition 1: Giá nằm dưới Mây
                cloud_bottom = min(SpanA[i], SpanB[i])
                if C[i] < cloud_bottom:
                    # Condition 2: Pullback chạm Kijun (High chạm Kijun)
                    pullback_kijun = (H[i] >= Kijun[i]) and (C[i] < Kijun[i])

                    if pullback_kijun:
                        position = -1
                        entry_price = C[i] * (1 - c['SLIPPAGE'])
                        # SL = Đỉnh mây + 1 ATR
                        cloud_top = max(SpanA[i], SpanB[i])
                        sl_price = cloud_top + (c['SL_ATR_MULT'] * ATR[i])

        equity_curve[i] = capital

    return pd.Series(equity_curve, index=df.index), pd.DataFrame(trades)

# ==============================================================================
# BƯỚC 6: MAIN EXECUTION
# ==============================================================================
if __name__ == "__main__":
    # 1. Tải Data 1H (Để trade)
    df_1h = fetch_binance_kline_df(CONFIG['SYMBOL'], CONFIG['INTERVAL'], CONFIG['START_DATE'], CONFIG['END_DATE'])

    if not df_1h.empty:
        # 2. Resample sang 4H để train HMM
        # Rule Resample: Open lấy đầu, High lấy max, Low lấy min, Close lấy cuối
        df_4h = df_1h.resample(CONFIG['HTF_INTERVAL']).agg({
            'Open': 'first', 'High': 'max', 'Low': 'min', 'Close': 'last', 'Volume': 'sum'
        }).dropna()

        # 3. Train HMM trên khung 4H
        df_hmm = train_hmm_rolling(df_4h, window_size=CONFIG['HMM_WINDOW'])

        # 4. Chạy Strategy Hybrid (1H + State từ 4H)
        equity, trades = run_hybrid_strategy(df_1h, df_hmm)

        # 5. Report
        if not trades.empty:
            final = equity.iloc[-1]
            profit = (final - CONFIG['INITIAL_CAPITAL']) / CONFIG['INITIAL_CAPITAL'] * 100
            dd = (equity - equity.cummax()) / equity.cummax()
            print(f"\n📊 KẾT QUẢ: HMM + ICHIMOKU")
            print(f"Vốn cuối: ${final:,.2f} ({profit:.2f}%)")
            print(f"Số lệnh: {len(trades)}")
            print(f"Max DD: {dd.min()*100:.2f}%")

            plt.figure(figsize=(12,6))
            plt.plot(equity, label='Equity'); plt.legend(); plt.show()
        else:
            print("⚠️ Không có lệnh nào được khớp.")